# Module 7 practice: extend the ShopKart pipeline (solutions)

**Practice after Video 7.2** | ShopKart project

**Goal:** Add a quality check, a new dimension table and a second incremental load to the pipeline you built in 7.2.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `shopkart_pipeline.py`, `orders_raw.csv`, `customers.csv`, `products.csv`, `new_orders_2026_01.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

We **import** the functions from `shopkart_pipeline.py` instead of copying them. This is how real pipelines reuse code.

In [1]:
import sqlite3
import pandas as pd
from shopkart_pipeline import extract, transform, quality_checks, load, incremental_load, table_sizes, WAREHOUSE_DB

# Build the warehouse first (same as Video 7.2)
orders, customers, products = extract()
clean = transform(orders, customers, products)
quality_checks(clean, expected_rows=orders["order_id"].nunique())
load(clean, customers, products)
incremental_load("new_orders_2026_01.csv")
table_sizes()

05:22:31  EXTRACT  orders_raw.csv: 5055 rows | customers: 800 | products: 60
05:22:31  TRANSFORM  5055 raw rows -> 5000 clean rows, 21 columns
    PASS - row count = unique orders in source
    PASS - no duplicate order_id
    PASS - no missing keys (order_id, customer_id, product_id, order_date)
    PASS - every product found in products.csv
    PASS - every customer found in customers.csv
    PASS - unit_price > 0
    PASS - quantity between 1 and 10
05:22:31  QUALITY CHECKS  all 7 passed
05:22:31  LOAD  shopkart_warehouse.db: fact_orders 5000, dim_customer 800, dim_product 60, dim_date 336 rows | shopkart_clean.csv (5000, 21)
05:22:31  INCREMENTAL LOAD  starting with new_orders_2026_01.csv
05:22:31  EXTRACT  new_orders_2026_01.csv: 40 rows | customers: 800 | products: 60
05:22:31  TRANSFORM  40 raw rows -> 38 clean rows, 21 columns
    PASS - row count = unique orders in source
    PASS - no duplicate order_id
    PASS - no missing keys (order_id, customer_id, product_id, order_date

05:22:31  INCREMENTAL LOAD  40 rows in file, 2 duplicate rows removed | added 36 new orders | skipped 2 already in warehouse | added 21 new dates


{'fact_orders': 5036, 'dim_customer': 800, 'dim_product': 60, 'dim_date': 357}

## Task 1 – One more quality check
Write `extra_checks(df)` that fails if any discount is below 0 or above 0.5, or if any rating (ignoring blanks) is outside 1–5. Run it on `clean` (should pass) and on a broken copy where one discount is 0.9 (should stop with a clear error).

In [2]:
def extra_checks(df):
    checks = {
        "discount between 0 and 0.5": df["discount"].between(0, 0.5).all(),
        "rating 1-5 or blank": df["rating"].dropna().between(1, 5).all(),
    }
    failed = [name for name, ok in checks.items() if not ok]
    for name, ok in checks.items():
        print("   ", "PASS" if ok else "FAIL", "-", name)
    if failed:
        raise ValueError("Data quality check failed: " + "; ".join(failed))

extra_checks(clean)

broken = clean.copy()
broken.loc[0, "discount"] = 0.9
try:
    extra_checks(broken)
except ValueError as error:
    print("PIPELINE STOPPED ->", error)

    PASS - discount between 0 and 0.5
    PASS - rating 1-5 or blank
    FAIL - discount between 0 and 0.5
    PASS - rating 1-5 or blank
PIPELINE STOPPED -> Data quality check failed: discount between 0 and 0.5


## Task 2 – A new dimension table: dim_city
Create `dim_city` with one row per city: `city`, `state`, `zone` (North / South / East / West). Load it into the warehouse, then use SQL to get **2025 revenue by zone**.

In [3]:
dim_city = pd.DataFrame({
    "city":  ["Delhi", "Lucknow", "Jaipur", "Mumbai", "Pune", "Ahmedabad", "Bengaluru", "Hyderabad", "Chennai", "Kolkata"],
    "state": ["Delhi", "Uttar Pradesh", "Rajasthan", "Maharashtra", "Maharashtra", "Gujarat",
              "Karnataka", "Telangana", "Tamil Nadu", "West Bengal"],
    "zone":  ["North", "North", "North", "West", "West", "West", "South", "South", "South", "East"],
})

conn = sqlite3.connect(WAREHOUSE_DB)
dim_city.to_sql("dim_city", conn, if_exists="replace", index=False)

# every city in fact_orders must exist in dim_city (otherwise the JOIN would drop orders)
missing = pd.read_sql("SELECT DISTINCT city FROM fact_orders WHERE city NOT IN (SELECT city FROM dim_city)", conn)
print("Cities missing from dim_city:", len(missing))

zone_revenue = pd.read_sql("""
SELECT c.zone, COUNT(*) AS orders, ROUND(SUM(f.revenue), 2) AS revenue
FROM fact_orders AS f
JOIN dim_city AS c ON f.city = c.city
JOIN dim_date AS d ON f.date_key = d.date_key
WHERE d.year = 2025
GROUP BY c.zone
ORDER BY revenue DESC
""", conn)
conn.close()
zone_revenue

Cities missing from dim_city: 0


,zone,orders,revenue
0,South,1673,5084788.90
1,West,1513,4542947.35
2,North,1400,3839709.25
3,East,414,1237204.20


## Task 3 – Prime vs Free in January 2026
Write a SQL query that joins `fact_orders`, `dim_customer` and `dim_date` and shows orders and revenue by membership for January 2026 only.

In [4]:
conn = sqlite3.connect(WAREHOUSE_DB)
jan = pd.read_sql("""
SELECT c.membership, COUNT(*) AS orders, ROUND(SUM(f.revenue), 2) AS revenue
FROM fact_orders AS f
JOIN dim_customer AS c ON f.customer_id = c.customer_id
JOIN dim_date     AS d ON f.date_key    = d.date_key
WHERE d.year = 2026 AND d.month = 1
GROUP BY c.membership
""", conn)
conn.close()
jan

,membership,orders,revenue
0,Free,21,99065.90
1,Prime,15,74083.25


## Task 4 – A second new-orders file
The shop sends a late file `new_orders_2026_02.csv`. Make it in code: take the first 5 rows of `new_orders_2026_01.csv`; give rows 3–5 the new order_ids O15037, O15038, O15039 and the date 2026-02-02; leave rows 1–2 as they are (they are old 2025 orders). Save it, run `incremental_load` on it and explain the numbers it prints.

In [5]:
feb = pd.read_csv("new_orders_2026_01.csv").head(5).copy()
feb.loc[2:4, "order_id"] = ["O15037", "O15038", "O15039"]
feb.loc[2:4, "order_date"] = "2026-02-02"
feb.to_csv("new_orders_2026_02.csv", index=False)
print(feb[["order_id", "order_date", "city", "unit_price"]])

added, skipped = incremental_load("new_orders_2026_02.csv")
print("Added:", added, "| Skipped:", skipped)
table_sizes()

  order_id  order_date       city unit_price
0   O14999  2025-12-28     Mumbai        596
1   O15000  2025-12-28    Lucknow       2499
2   O15037  2026-02-02     Mumbai        254
3   O15038  2026-02-02  Hyderabad      19423
4   O15039  2026-02-02      delhi        407
05:22:31  INCREMENTAL LOAD  starting with new_orders_2026_02.csv
05:22:31  EXTRACT  new_orders_2026_02.csv: 5 rows | customers: 800 | products: 60
05:22:31  TRANSFORM  5 raw rows -> 5 clean rows, 21 columns
    PASS - row count = unique orders in source
    PASS - no duplicate order_id
    PASS - no missing keys (order_id, customer_id, product_id, order_date)
    PASS - every product found in products.csv
    PASS - every customer found in customers.csv
    PASS - unit_price > 0
    PASS - quantity between 1 and 10
05:22:31  QUALITY CHECKS  all 7 passed
05:22:32  INCREMENTAL LOAD  5 rows in file, 0 duplicate rows removed | added 3 new orders | skipped 2 already in warehouse | added 1 new dates
Added: 3 | Skipped: 2


{'fact_orders': 5039, 'dim_customer': 800, 'dim_product': 60, 'dim_date': 358}

**Answer:** 3 new orders are added (O15037–O15039) and 2 are skipped (O14999 and O15000 were loaded in 2025). One new date (2026-02-02) is added to `dim_date`. Nothing is duplicated, because `incremental_load` only appends order_ids the warehouse has not seen.